# 02 — Avaliando o modelo (teste de realidade)

Prever o futuro não pode ser conferido na hora. A saída é **esconder o final do histórico**, prever só com o que sobrou e comparar com o que de fato aconteceu.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from trafego_lab.backtest import run_backtest
from trafego_lab.dados import detect_columns, prepare_series, resample_series
from trafego_lab.modelo import forecast, load_forecaster

df = pd.read_csv(Path('..') / 'data' / 'samples' / 'trafego_sintetico.csv')
date_col, value_col = detect_columns(df)
series = resample_series(prepare_series(df, date_col, value_col), 'h', how='sum')
model = load_forecaster()

## Rodar o teste

Escondemos as últimas 48 horas. O parâmetro `season=24` define a referência simples: repetir as últimas 24 horas.

In [ ]:
horizon = 48
bt = run_backtest(series, horizon, lambda hist, n: forecast(model, hist, n), season=24)

print(f'MAE TimesFM:          {bt.mae:,.1f} veículos/h')
print(f'MAE repetir ciclo:    {bt.baseline_mae:,.1f} veículos/h')
print(f'MAPE:                 {bt.mape:.1f}%')
print(f'Cobertura da faixa:   {bt.coverage:.0%} (ideal: perto de 80%)')

## Como ler

- **MAE**: erro médio em veículos por hora. Menor é melhor.
- Se o MAE do TimesFM é menor que o de *repetir o ciclo*, o modelo agrega valor.
- **Cobertura** bem abaixo de 80% significa que a faixa de incerteza é otimista demais.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
bt.history.iloc[-24 * 4:].plot(ax=ax, color='gray', label='Histórico')
bt.actual.plot(ax=ax, color='black', label='Real')
bt.baseline.plot(ax=ax, color='lightgray', ls=':', label='Repetir ciclo')
bt.point.plot(ax=ax, color='tab:blue', lw=2, label='TimesFM')
ax.fill_between(bt.point.index, bt.lower, bt.upper, alpha=0.2)
ax.legend()
plt.show()

## Erro por hora do dia

Onde o modelo erra mais: nos picos ou fora deles?

In [ ]:
errors = (bt.actual - bt.point).abs()
errors.groupby(errors.index.hour).mean().plot(kind='bar', figsize=(10, 3), title='Erro médio por hora do dia')
plt.show()

## Para experimentar

- Aumente o `horizon` para 168 (uma semana) e use `season=168`.
- Rode o mesmo teste num dataset real do Kaggle. Onde o erro cresce? Em feriados?